In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import anndata as ad
from collections import defaultdict

import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
import seaborn as sns

import scipy as sp
from scipy import stats as st
import statsmodels.stats.anova as anova 
from statsmodels.stats.multicomp import pairwise_tukeyhsd
#from statannot import add_stat_annotation

import openpyxl
import yaml
from pathlib import Path

In [ ]:
sc.set_figure_params(dpi=100, color_map = 'viridis_r')
sc.settings.verbosity = 1
sc.logging.print_header()

In [ ]:
files = ['/data/projects/manuela/scRNA_aging-mouse/sub2_integrated-aging-soupxed.h5ad',
        '/data/projects/manuela/scRNA_aging-mouse/sub3_integrated-aging-soupxed.h5ad',
        '/data/projects/manuela/scRNA_aging-mouse/sub4_integrated-aging-soupxed.h5ad',
        '/data/projects/manuela/scRNA_aging-mouse/sub12_integrated-aging-soupxed.h5ad',
        '/data/projects/manuela/scRNA_aging-mouse/sub13_integrated-aging-soupxed.h5ad'
                   ]

In [ ]:
adata = sc.read("/data/projects/manuela/scRNA_aging-mouse/integrated-aging-soupxed.h5ad", var_names="gene_symbols")
sub_data = ad.concat({file:sc.read_h5ad(file) for file in files})

sub2 = sc.read("/data/projects/manuela/scRNA_aging-mouse/sub2_integrated-aging-soupxed.h5ad", var_names="gene_symbols")
sub3 = sc.read("/data/projects/manuela/scRNA_aging-mouse/sub3_integrated-aging-soupxed.h5ad", var_names="gene_symbols")
sub4 = sc.read("/data/projects/manuela/scRNA_aging-mouse/sub4_integrated-aging-soupxed.h5ad", var_names="gene_symbols")
sub12 = sc.read("/data/projects/manuela/scRNA_aging-mouse/sub12_integrated-aging-soupxed.h5ad", var_names="gene_symbols")
sub13 = sc.read("/data/projects/manuela/scRNA_aging-mouse/sub13_integrated-aging-soupxed.h5ad", var_names="gene_symbols")

In [ ]:
cluster = {"/data/projects/manuela/scRNA_aging-mouse/sub2_integrated-aging-soupxed.h5ad": "2",
      "/data/projects/manuela/scRNA_aging-mouse/sub3_integrated-aging-soupxed.h5ad": "3",
       "/data/projects/manuela/scRNA_aging-mouse/sub4_integrated-aging-soupxed.h5ad": "4",
       "/data/projects/manuela/scRNA_aging-mouse/sub12_integrated-aging-soupxed.h5ad": "12",
       "/data/projects/manuela/scRNA_aging-mouse/sub13_integrated-aging-soupxed.h5ad": "13"
    }
sub_data.obs['clusters'] = sub_data.obs['clusters'].replace(cluster)

# concatenate cluster and subcluster columns into a new column
sub_data.obs['mergedclusters'] = sub_data.obs['clusters'] + '.' + sub_data.obs['subclusters']
#sub_data.obs

In [ ]:
#sc.tl.leiden(adata, key_added='clusters', resolution=0.31)

#rc_context is used for the figure size, in this case 4x4
adata.obs.clusters = adata.obs.clusters.astype('category')
with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.umap(adata, color=['clusters'], wspace=0.4)

In [ ]:
adata.obs.clusters = adata.obs.clusters.astype('category')
with rc_context({'figure.figsize': (4, 4)}):
    sc.pl.umap(sub_data, color=['clusters', 'mergedclusters'], wspace=0.4)

## Calculate Cell Counts per Cluster & per Sample per Cluster

## Create kidney Celltype Marker Gene List

In [ ]:
#import celltype marker list
celltypemarkers_dict = yaml.safe_load(Path('/data/projects/manuela/bulkRNA_mouse/mouse_markers_detailed.yaml').read_text())
subset_dict = {k: [i for i in v if i in adata.raw.var_names.values.tolist()] for k, v in celltypemarkers_dict.items() if any(i in v for i in adata.raw.var_names.values.tolist())}

In [ ]:
subset_dict

In [ ]:
# create list with all celltypemarker genes
celltypemarkers_list_notflat = subset_dict.values()
celltypemarkers_list = [item for sublist in celltypemarkers_list_notflat for item in sublist]
#print(celltypemarkers_list)

### Gene Expressin UMAPs for each cell type

In [ ]:
for ct in subset_dict:
   # print(f"{ct.upper()}:")  # print cell subtype name
    print(ct) 
    sc.pl.umap(
        adata,
        color=subset_dict[ct],
        vmin=0,
        vmax="p99",  # set vmax to the 99th percentile of the gene count instead of the maximum, to prevent outliers from making expression in other cells invisible. Note that this can cause problems for extremely lowly expressed genes.
        sort_order=False,  # do not plot highest expression on top, to not get a biased view of the mean expression among cells
        frameon=False,
        cmap="Reds",  # or choose another color map e.g. from here: https://matplotlib.org/stable/tutorials/colors/colormaps.html
        save = '_'+ct
    )
    print("\n\n\n")  # print white space for 

### Dotplots

In [ ]:
#sc.pl.DotPlot(adata, use_raw=True, var_names=celltypemarkers_sub, groupby='subclusters', title='Celltypemarkers genes expression in clusters ', cmap='BuGn').savefig('appr1-subclutser-dotplot_pig.png')
sc.pl.DotPlot(sub2, use_raw=True, var_names=subset_dict, groupby='subclusters', standard_scale="var", title='Subcluster 2-Celltypemarkers genes expression in clusters ', cmap='BuGn').savefig('subclutser2-dotplot_aging.png')

In [ ]:
sc.pl.DotPlot(sub3, use_raw=True, var_names=subset_dict, groupby='subclusters', standard_scale="var", title='Subcluster 3-Celltypemarkers genes expression in clusters ', cmap='BuGn').savefig('subclutser3-dotplot_aging.png')

In [ ]:
sc.pl.DotPlot(sub4, use_raw=True, var_names=subset_dict, groupby='subclusters', standard_scale="var", title='Subcluster 4-Celltypemarkers genes expression in clusters ', cmap='BuGn').savefig('subclutser4-dotplot_aging.png')

In [ ]:
sc.pl.DotPlot(sub12, use_raw=True, var_names=subset_dict, groupby='subclusters', standard_scale="var", title='Subcluster 12-Celltypemarkers genes expression in clusters ', cmap='BuGn').savefig('subclutser12-dotplot_aging.png')

In [ ]:
sc.pl.DotPlot(sub13, use_raw=True, var_names=subset_dict, groupby='subclusters', standard_scale="var", title='Subcluster 13-Celltypemarkers genes expression in clusters ', cmap='BuGn').savefig('subclutser13-dotplot_aging.png')

## DE Analysis by Clusters

In [ ]:
sub2.uns['log1p']["base"] = None
sub3.uns['log1p']["base"] = None
sub4.uns['log1p']["base"] = None
sub12.uns['log1p']["base"] = None
sub13.uns['log1p']["base"] = None

sc.tl.rank_genes_groups(sub2, "subclusters", method='t-test')
sc.tl.rank_genes_groups(sub3, "subclusters", method='t-test')
sc.tl.rank_genes_groups(sub4, "subclusters", method='t-test')
sc.tl.rank_genes_groups(sub12, "subclusters", method='t-test')
sc.tl.rank_genes_groups(sub13, "subclusters", method='t-test')

#sc.pl.rank_genes_groups(sub2, n_genes=10, sharey=True)
#sc.pl.rank_genes_groups(sub3, n_genes=10, sharey=True)
#sc.pl.rank_genes_groups(sub4, n_genes=10, sharey=True)
#sc.pl.rank_genes_groups(sub12, n_genes=10, sharey=True)
#sc.pl.rank_genes_groups(sub13, n_genes=10, sharey=True)

### Dotplots Top 5 DEGs between Clusters

### Intersection of Top 10/20 DEGs per Cluster with Reference Marker List

In [ ]:
# Get Top DE genes per cluster
result = sub2.uns['rank_genes_groups']
groups = result['names'].dtype.names
top_genes_per_cluster = {}
for group in groups:
    top_genes_per_cluster[group] = result['names'][group][:30]

# Get all unique genes across all clusters
sub2_top_genes = set().union(*top_genes_per_cluster.values())
#all_top_genes

In [ ]:
result = sub3.uns['rank_genes_groups']
groups = result['names'].dtype.names
top_genes_per_cluster = {}
for group in groups:
    top_genes_per_cluster[group] = result['names'][group][:30]

# Get all unique genes across all clusters
sub3_top_genes = set().union(*top_genes_per_cluster.values())

In [ ]:
result = sub4.uns['rank_genes_groups']
groups = result['names'].dtype.names
top_genes_per_cluster = {}
for group in groups:
    top_genes_per_cluster[group] = result['names'][group][:30]

# Get all unique genes across all clusters
sub4_top_genes = set().union(*top_genes_per_cluster.values())

In [ ]:
result = sub12.uns['rank_genes_groups']
groups = result['names'].dtype.names
top_genes_per_cluster = {}
for group in groups:
    top_genes_per_cluster[group] = result['names'][group][:30]

# Get all unique genes across all clusters
sub12_top_genes = set().union(*top_genes_per_cluster.values())

In [ ]:
result = sub13.uns['rank_genes_groups']
groups = result['names'].dtype.names
top_genes_per_cluster = {}
for group in groups:
    top_genes_per_cluster[group] = result['names'][group][:30]

# Get all unique genes across all clusters
sub13_top_genes = set().union(*top_genes_per_cluster.values())

In [ ]:
reference_genes_df = pd.DataFrame(list(celltypemarkers_dict.items()), columns=['cell_type', 'marker_genes'])
reference_genes_df = reference_genes_df.explode('marker_genes')
# Reset the index of the dataframe
reference_genes_df = reference_genes_df.reset_index(drop=True)

In [ ]:
# Intersection of the reference markers and the top 10 genes per cluster
reference_genes = set(reference_genes_df['marker_genes'])
intersection_genes2 = reference_genes.intersection(sub2_top_genes)

reference_genes = set(reference_genes_df['marker_genes'])
intersection_genes3 = reference_genes.intersection(sub3_top_genes)

reference_genes = set(reference_genes_df['marker_genes'])
intersection_genes4 = reference_genes.intersection(sub4_top_genes)

reference_genes = set(reference_genes_df['marker_genes'])
intersection_genes12 = reference_genes.intersection(sub12_top_genes)

reference_genes = set(reference_genes_df['marker_genes'])
intersection_genes13 = reference_genes.intersection(sub13_top_genes)

In [ ]:
celltypemarkers_dict_subset2 = {}
for marker, genes in celltypemarkers_dict.items():
    subset_genes = [gene for gene in genes if gene in intersection_genes2]
    #print(subset_genes)
    if len(subset_genes) > 0:
        celltypemarkers_dict_subset2[marker] = subset_genes

In [ ]:
celltypemarkers_dict_subset3 = {}
for marker, genes in celltypemarkers_dict.items():
    subset_genes = [gene for gene in genes if gene in intersection_genes3]
    #print(subset_genes)
    if len(subset_genes) > 0:
        celltypemarkers_dict_subset3[marker] = subset_genes

In [ ]:
celltypemarkers_dict_subset4 = {}
for marker, genes in celltypemarkers_dict.items():
    subset_genes = [gene for gene in genes if gene in intersection_genes4]
    #print(subset_genes)
    if len(subset_genes) > 0:
        celltypemarkers_dict_subset4[marker] = subset_genes

In [ ]:
celltypemarkers_dict_subset12 = {}
for marker, genes in celltypemarkers_dict.items():
    subset_genes = [gene for gene in genes if gene in intersection_genes12]
    #print(subset_genes)
    if len(subset_genes) > 0:
        celltypemarkers_dict_subset12[marker] = subset_genes

In [ ]:
celltypemarkers_dict_subset13 = {}
for marker, genes in celltypemarkers_dict.items():
    subset_genes = [gene for gene in genes if gene in intersection_genes13]
    #print(subset_genes)
    if len(subset_genes) > 0:
        celltypemarkers_dict_subset13[marker] = subset_genes

### Dotplots of Intersected DEGs from Reference Marker List 

#### Subclusters cluster 2

In [ ]:
# Dotplot with celltypemarkers and celltypes from marker gene disctionary
sc.pl.DotPlot(sub2, use_raw=True, var_names=celltypemarkers_dict_subset2, groupby='subclusters', title='Celltypemarkers genes expression in subclusters of cluster 2', cmap='BuGn').savefig('dotplot_sub2_agingMouse.png')

Setting standard_scale="var" will scale each gene's expression values by subtracting the mean and dividing by the standard deviation of that gene across cells. This can help to highlight genes that are consistently expressed across cells, regardless of the absolute expression levels.

In [ ]:
# Dotplot with celltypemarkers and celltypes from marker gene disctionary
sc.pl.DotPlot(sub2, use_raw=True, standard_scale="var", var_names=celltypemarkers_dict_subset2, groupby='subclusters', title='Celltypemarkers genes expression in subclusters of cluster 2', cmap='BuGn').savefig('dotplot_var_sub2_agingMouse.png')

#### Subclusters cluster 3

In [ ]:
# Dotplot with celltypemarkers and celltypes from marker gene disctionary
sc.pl.DotPlot(sub3, use_raw=True, var_names=celltypemarkers_dict_subset3, groupby='subclusters', title='Celltypemarkers genes expression in subclusters of cluster 3', cmap='BuGn').savefig('dotplot_sub3_agingMouse.png')

In [ ]:
# Dotplot with celltypemarkers and celltypes from marker gene disctionary
sc.pl.DotPlot(sub3, use_raw=True, var_names=celltypemarkers_dict_subset3, groupby='subclusters', standard_scale="var", title='Celltypemarkers genes expression in subclusters of cluster 3', cmap='BuGn').savefig('dotplot_var_sub3_agingMouse.png')

#### Subclusters cluster 4

In [ ]:
# Dotplot with celltypemarkers and celltypes from marker gene disctionary
sc.pl.DotPlot(sub4, use_raw=True, var_names=celltypemarkers_dict_subset4, groupby='subclusters', title='Celltypemarkers genes expression in subclusters of cluster 4', cmap='BuGn').savefig('dotplot_sub4_agingMouse.png')

In [ ]:
# Dotplot with celltypemarkers and celltypes from marker gene disctionary
sc.pl.DotPlot(sub4, use_raw=True, var_names=celltypemarkers_dict_subset4, groupby='subclusters', standard_scale="var", title='Celltypemarkers genes expression in subclusters of cluster 4', cmap='BuGn').savefig('dotplot_var_sub4_agingMouse.png')

In [ ]:
sc.pl.rank_genes_groups_dotplot(sub4, groupby="subclusters", standard_scale="var", var_names=celltypemarkers_dict_subset4 , title='Top 5 Genes expression in subclusters of cluster 4', key="rank_genes_groups")

#### Subclusters cluster 12

In [ ]:
# Dotplot with celltypemarkers and celltypes from marker gene disctionary
sc.pl.DotPlot(sub12, use_raw=True, var_names=celltypemarkers_dict_subset12, groupby='subclusters', title='Celltypemarkers genes expression in subclusters of cluster 12', cmap='BuGn').savefig('dotplot_sub12_agingMouse.png')

In [ ]:
# Dotplot with celltypemarkers and celltypes from marker gene disctionary
sc.pl.DotPlot(sub12, use_raw=True, var_names=celltypemarkers_dict_subset12, groupby='subclusters', standard_scale="var", title='Celltypemarkers genes expression in subclusters of cluster 12', cmap='BuGn').savefig('dotplot_var_sub12_agingMouse.png')

In [ ]:
sc.pl.rank_genes_groups_dotplot(sub12, groupby="subclusters", standard_scale="var", var_names=celltypemarkers_dict_subset12 , title='Top 5 Genes expression in subclusters of cluster 12', key="rank_genes_groups").savefig('dotplot_sub12_var_agingMouse.png')

#### Subclusters cluster 13

In [ ]:
# Dotplot with celltypemarkers and celltypes from marker gene disctionary
sc.pl.DotPlot(sub13, use_raw=True, var_names=celltypemarkers_dict_subset13, groupby='subclusters', title='Celltypemarkers genes expression in subclusters of cluster 13', cmap='BuGn').savefig('dotplot_sub13_agingMouse.png')

In [ ]:
# Dotplot with celltypemarkers and celltypes from marker gene disctionary
sc.pl.DotPlot(sub13, use_raw=True, var_names=celltypemarkers_dict_subset13, groupby='subclusters', standard_scale="var", title='Celltypemarkers genes expression in subclusters of cluster 13', cmap='BuGn').savefig('dotplot_var_sub13_agingMouse.png')

In [ ]:
sc.pl.rank_genes_groups_dotplot(sub13, groupby="subclusters", standard_scale="var", var_names=celltypemarkers_dict_subset13 , title='Top 5 Genes expression in subclusters of cluster 13', key="rank_genes_groups")

## Excel Tables

In [ ]:
sub2_cluster_gene_table = sc.get.rank_genes_groups_df(sub2, group= None)
sub3_cluster_gene_table = sc.get.rank_genes_groups_df(sub3, group= None)
sub4_cluster_gene_table = sc.get.rank_genes_groups_df(sub4, group= None)
sub12_cluster_gene_table = sc.get.rank_genes_groups_df(sub12, group= None)
sub13_cluster_gene_table = sc.get.rank_genes_groups_df(sub13, group= None)

#sub2_cluster_gene_table

In [ ]:
sub2_df = pd.pivot_table(data=sub2_cluster_gene_table, index=['names'], columns=['group'])
sub3_df = pd.pivot_table(data=sub3_cluster_gene_table, index=['names'], columns=['group'])
sub4_df = pd.pivot_table(data=sub4_cluster_gene_table, index=['names'], columns=['group'])
sub12_df = pd.pivot_table(data=sub12_cluster_gene_table, index=['names'], columns=['group'])
sub13_df = pd.pivot_table(data=sub13_cluster_gene_table, index=['names'], columns=['group'])

In [ ]:
with pd.ExcelWriter("/data/projects/manuela/scRNA_aging-mouse/aging_Subcluster2_Markers.xlsx", engine='openpyxl') as writer:
    for clustername in sub2_df.columns.levels[1]:
        sub2_df.xs(clustername, level="group", axis=1).to_excel(writer, sheet_name=f"Cluster_{clustername}_vs. rest")

In [ ]:
with pd.ExcelWriter("/data/projects/manuela/scRNA_aging-mouse/aging_Subcluster3_Markers.xlsx", engine='openpyxl') as writer:
    for clustername in sub3_df.columns.levels[1]:
        sub3_df.xs(clustername, level="group", axis=1).to_excel(writer, sheet_name=f"Cluster_{clustername}_vs. rest")

In [ ]:
with pd.ExcelWriter("/data/projects/manuela/scRNA_aging-mouse/aging_Subcluster4_Markers.xlsx", engine='openpyxl') as writer:
    for clustername in sub4_df.columns.levels[1]:
        sub4_df.xs(clustername, level="group", axis=1).to_excel(writer, sheet_name=f"Cluster_{clustername}_vs. rest")

In [ ]:
with pd.ExcelWriter("/data/projects/manuela/scRNA_aging-mouse/aging_Subcluster12_Markers.xlsx", engine='openpyxl') as writer:
    for clustername in sub12_df.columns.levels[1]:
        sub12_df.xs(clustername, level="group", axis=1).to_excel(writer, sheet_name=f"Cluster_{clustername}_vs. rest")

In [ ]:
with pd.ExcelWriter("/data/projects/manuela/scRNA_aging-mouse/aging_Subcluster13_Markers.xlsx", engine='openpyxl') as writer:
    for clustername in sub13_df.columns.levels[1]:
        sub13_df.xs(clustername, level="group", axis=1).to_excel(writer, sheet_name=f"Cluster_{clustername}_vs. rest")